# Dataset Exploration
Inspect raw inference logs and curated datasets. Understand what the curation pipeline keeps, drops, and why.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from elasticsearch import Elasticsearch
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv('../.env')

es = Elasticsearch(os.getenv('ES_HOST', 'http://localhost:9200'))
mongo = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017'))
db = mongo[os.getenv('MONGO_DB', 'data_flywheel')]

print('Connected to Elasticsearch and MongoDB')

## 1. Raw Inference Logs

In [ ]:
# Pull all logs from Elasticsearch
resp = es.search(index=os.getenv('ES_INDEX_LOGS', 'inference_logs'), body={'query': {'match_all': {}}, 'size': 5000})
logs = [hit['_source'] for hit in resp['hits']['hits']]
df_logs = pd.DataFrame(logs)

print(f'Total logs: {len(df_logs)}')
print(f'Curated:    {df_logs["curated_in_run"].notna().sum() if "curated_in_run" in df_logs else 0}')
print(f'Uncurated:  {df_logs["curated_in_run"].isna().sum() if "curated_in_run" in df_logs else len(df_logs)}')
df_logs.head()

In [ ]:
# Response length distribution
df_logs['response_words'] = df_logs['response'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_logs['response_words'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Response Length Distribution (words)')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')

df_logs['latency_ms'].hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Latency Distribution (ms)')
axes[1].set_xlabel('Latency (ms)')

plt.tight_layout()
plt.show()

## 2. Curation Filter Analysis

In [ ]:
from orchestrator.services.curator.filters import FilterPipeline

pipeline = FilterPipeline(
    min_quality_score=0.7,
    max_token_length=2048,
    min_response_length=20,
    remove_pii=False,
)

samples = df_logs[['prompt', 'response', 'model', 'latency_ms']].to_dict('records')

# Run each filter individually to count drop reasons
drop_reasons = {}
kept = []
for s in samples:
    reason = pipeline._filter(s)
    if reason:
        drop_reasons[reason] = drop_reasons.get(reason, 0) + 1
    else:
        kept.append(s)

print(f'Input:  {len(samples)}')
print(f'Kept:   {len(kept)} ({len(kept)/len(samples)*100:.1f}%)')
print(f'Dropped: {len(samples)-len(kept)}')
print(f'\nDrop reasons: {drop_reasons}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Drop reasons bar chart
if drop_reasons:
    axes[0].bar(drop_reasons.keys(), drop_reasons.values(), color='salmon', edgecolor='white')
    axes[0].set_title('Drop Reasons')
    axes[0].set_ylabel('Count')

# Keep vs drop pie
axes[1].pie(
    [len(kept), len(samples) - len(kept)],
    labels=['Kept', 'Dropped'],
    colors=['steelblue', 'salmon'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[1].set_title('Filter Outcome')

plt.tight_layout()
plt.show()

## 3. Quality Score Distribution

In [ ]:
scores = []
for s in samples:
    text = f"{s.get('prompt', '')} {s.get('response', '')}"
    score = pipeline._quality_score(text)
    scores.append(score)

df_logs['quality_score'] = scores

plt.figure(figsize=(10, 4))
plt.hist(scores, bins=40, color='steelblue', edgecolor='white')
plt.axvline(0.7, color='red', linestyle='--', label='Min threshold (0.7)')
plt.title('Quality Score Distribution')
plt.xlabel('Quality score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean: {pd.Series(scores).mean():.3f}')
print(f'Below threshold: {sum(1 for s in scores if s < 0.7)} ({sum(1 for s in scores if s < 0.7)/len(scores)*100:.1f}%)')

## 4. Curated Datasets

In [ ]:
datasets = list(db.datasets.find({}, {'_id': 1, 'run_id': 1, 'created_at': 1, 'sample_count': 1}))
df_datasets = pd.DataFrame(datasets)

if len(df_datasets):
    print(f'Total datasets: {len(df_datasets)}')
    print(f'Total curated samples: {df_datasets["sample_count"].sum()}')
    display(df_datasets)
else:
    print('No curated datasets yet — run: make run-flywheel')